### KNNとは
- K近傍法（K-Nearest Neighbors）は、予測したいデータに最も近いK個のデータを探し、その多数決でクラスを決める手法
- 「似たものを同じクラスにする」という直感的な考え方に基づく
- 学習フェーズでは何も計算せず、予測のタイミングで初めて距離計算が走る。（遅延学習（Lazy Learning））

||内容|
|----|----|
|モデル|訓練データそのもの（パラメータなし）|
|距離指標|ユークリッド距離|
|予測|K個の近傍の多数決|
|学習コスト|ほぼゼロ（データを記憶するだけ）|
|予測コスト|高い（全訓練データとの距離計算が必要）|



KNNは学習が不要な分、
予測のたびに全訓練データとの距離計算が走る。

データ量が増えるほど予測が遅くなり、大規模なデータには向かない。

In [ ]:
# ライブラリでのKNNの実装例
# 一般的にkのチューニングは、交差検証を使用する。

import numpy as np
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


In [2]:
# scratchでのKNNの実装例

import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


class KNeighborsClassifier:
    def __init__(self, n_neighbors=3):
        self.n_neighbors = n_neighbors
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _euclidean_distance(self, x):
        return np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))

    def _predict_one(self, x):
        distances = self._euclidean_distance(x)

        k_indices = np.argsort(distances)[:self.n_neighbors]
        k_labels = self.y_train[k_indices]

        return np.bincount(k_labels).argmax()

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])


# 実行
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"正解率: {accuracy_score(y_test, y_pred):.4f}")

正解率: 1.0000


`fit` メソッド

訓練データをそのまま保存するだけで、決定木やロジスティック回帰と違い、パラメータの最適化は一切行いません。

`_euclidean_distance` メソッド

`self.X_train - x` はブロードキャストにより、訓練データ全点と `x` の差を一度に計算することができます。`axis=1` で各データ点ごとに次元方向に足し合わせ、`sqrt` でユークリッド距離が得られます。

`_predict_one` メソッド

`np.argsort` は配列を昇順に並べたときのインデックスを返します。先頭K個が最近傍です。`np.bincount` はラベルの出現回数を数えるため、`.argmax()` で最頻クラスが得られます。